# Validate Step 2.3.1 — Global Projections (Cleaned Up) ✅

| | |
|---|---|
| **Input** | `output/step_2/yolo_global/<sample>/<camera>.json` (from new Step 2.3.1) |
| **Outputs** | Console report + `output/step_2/projection_validation_report.json` |

---

### What changed

- Removed duplicate validation cells (the original had the same check written twice, plus a markdown cell containing plain code text instead of an actual code cell).
- Path fixed to `STEP2_DIR / "yolo_global"`, matching the new Step 2.3.1 output.
- Added the **content-level check** your own notes said was still needed: not just "does the file exist," but "does it actually contain a usable 3D position."

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP2_DIR

PROJ_DIR = STEP2_DIR / "yolo_global"
if not PROJ_DIR.exists():
    raise FileNotFoundError(f"{PROJ_DIR} not found — run Step 2.3.1 first.")

print(f"✅ PROJ_DIR: {PROJ_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ PROJ_DIR: F:\Sensor fusion Research\output\step_2\yolo_global


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Two-level validation: folder/file presence AND content quality
# ─────────────────────────────────────────────────────────────────

import json

CAMERAS = {'CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
           'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT'}

sample_dirs = sorted([p for p in PROJ_DIR.glob("sample_*") if p.is_dir()])
print(f"✅ Total samples with projection folders: {len(sample_dirs)}\n")

incomplete_samples = []       # missing one or more camera JSON files
content_issues = []           # files exist but have zero 3D-matched detections

total_detections = 0
total_with_3d = 0

for sample_dir in sample_dirs:
    cams_present = {p.stem for p in sample_dir.glob("*.json")}
    missing = CAMERAS - cams_present
    if missing:
        incomplete_samples.append({"sample_id": sample_dir.name, "missing": sorted(missing)})
        continue

    # Content-level check — does each file have usable 3D positions?
    for cam_file in sample_dir.glob("*.json"):
        with open(cam_file) as f:
            dets = json.load(f)

        total_detections += len(dets)
        n_with_3d = sum(1 for d in dets if d.get("has_3d_position"))
        total_with_3d += n_with_3d

        if len(dets) > 0 and n_with_3d == 0:
            content_issues.append({"sample_id": sample_dir.name, "camera": cam_file.stem})

# ── Report ──
print(f"🚨 Samples missing camera files : {len(incomplete_samples)}")
if incomplete_samples:
    for entry in incomplete_samples[:10]:
        print(f"   - {entry['sample_id']}: missing {entry['missing']}")

print(f"\n🚨 Sample/camera combos with ZERO 3D-matched detections : {len(content_issues)}")
if content_issues:
    for entry in content_issues[:10]:
        print(f"   - {entry['sample_id']} / {entry['camera']}")

match_rate = total_with_3d / total_detections * 100 if total_detections > 0 else 0
print(f"\n✅ Overall 3D match rate across all detections: {match_rate:.1f}%")
print(f"   ({total_with_3d} / {total_detections} detections have a usable global 3D position)")

report = {
    "total_samples": len(sample_dirs),
    "incomplete_samples": incomplete_samples,
    "content_issues": content_issues,
    "overall_match_rate_pct": round(match_rate, 1)
}
report_path = STEP2_DIR / "projection_validation_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\n📄 Report saved: {report_path}")

✅ Total samples with projection folders: 404

🚨 Samples missing camera files : 0

🚨 Sample/camera combos with ZERO 3D-matched detections : 8
   - sample_0155 / CAM_FRONT
   - sample_0161 / CAM_FRONT
   - sample_0217 / CAM_FRONT
   - sample_0230 / CAM_FRONT
   - sample_0232 / CAM_FRONT
   - sample_0377 / CAM_FRONT_LEFT
   - sample_0380 / CAM_FRONT_LEFT
   - sample_0391 / CAM_FRONT_LEFT

✅ Overall 3D match rate across all detections: 91.9%
   (8365 / 9099 detections have a usable global 3D position)

📄 Report saved: F:\Sensor fusion Research\output\step_2\projection_validation_report.json
